In [1]:
import pandas as pd
import numpy as np
import MDAnalysis as mda
import matplotlib.pyplot as plt
from itertools import combinations
import sys
from MDAnalysis.analysis import contacts
from itertools import product
import itertools

In [2]:
path_tools = '/home/lubaltz/code/SFB1551/r08' 
sys.path.append(path_tools + '/GMX_models/py_tools')
sys.path.append(path_tools + '/GMX_models/signac')
#import toolbox_interactions_done as interactiontools
import toolbox_interactions_tested as interactiontools

In [3]:
def empty_contact_df():
    """Create an empty DataFrame with the correct schema for contacts."""
    meta_dict = {
        'frame': pd.Series([], dtype='int64'),
        'n_cont': pd.Series([], dtype='int64'),
        'cont_a': pd.Series([], dtype='object'),
        'cont_b': pd.Series([], dtype='object'),
        'q_values': pd.Series([], dtype='object'),
    }

    if bool_bonds:
        meta_dict.update({
            'cation_pi': pd.Series([], dtype='object'),
            'pi_stacking': pd.Series([], dtype='object'),
            'hbond': pd.Series([], dtype='object'),
            'salt_bridge': pd.Series([], dtype='object')
        })

    return pd.DataFrame(meta_dict)

In [4]:
bool_debug=False
bool_bonds=True
bool_q=False
def process_frame_chunk_for_split(top, traj, group_a_sel, group_b_sel, d_max, radius, k, chunk_size):
    """
    Process a chunk of frames to calculate contacts and specific interactions.
    
    Parameters
    ----------
    top, traj : str
    group_a_sel, group_b_sel : str
    d_max, radius : float
    k, chunk_size : int
    
    Returns
    -------
    pandas.DataFrame
    """

    #match = re.search(r'chunk_(\d+)', traj)
    #if match:
    #    # Convert 1-based file index to 0-based calculation index
    #    k = int(match.group(1)) - 1
    #else:
    #    print('check traj part', traj, flush=True)
        
    if bool_debug:
        worker_id = get_worker()
        print('part index',k, traj, flush=True)
        print('WORKER_ID', worker_id.name, flush=True)
        print(f"{worker_id.memory_manager.memory_limit / 1024**3:.2f} GB", flush=True)
        print(f"[{worker_id}] memory before processing {traj}: {psutil.Process().memory_info().rss / 1024**2:.2f} MB", flush=True)

    u = mda.Universe(top, traj)#, format='xtc', topology_format='pdb')

    str_a = group_a_sel + " and not name H*"
    u_group_a_sel = u.select_atoms(group_a_sel) 
    group_a = u_group_a_sel.select_atoms("not name H*")

    str_b = group_b_sel + " and not name H*"
    group_b = u.select_atoms(str_b) 
    
    if bool_bonds:
        group_a_sc_H = """
            ((resname SER and (name HG HG1)) or
            (resname THR and name HG1) or
            (resname TYR and name HH) or
            (resname ASN and (name HD21 HD22)) or
            (resname GLN and (name HE21 HE22)) or
            (resname HIS HSD HSE HSP and (name HD1 HE2)) or
            (resname TRP and name HE1) or
            (resname LYS and (name HZ1 HZ2 HZ3)) or
            (resname ARG and (name HE HH11 HH12 HH21 HH22)))
        """

        #for h-bonds - expand selection to hydrogens 
        #str_c=group_a_sel+ " and " +group_a_sc_H
        #group_c = u.select_atoms(str_c)
        #group_c=u_group_a_sel.select_atoms(group_a_sc_H)
        #print("c_selection", set(group_c.atoms.names), len(group_c.atoms.ids), flush=True)

        #selecting specific atoms participating in bonds
        cation_pi_selection = interactiontools.get_interaction_sel('cation_pi', group_a, group_b)
        pi_stacking_selection = interactiontools.get_interaction_sel('pi_stacking', group_a, group_b)
        #hbond_selection=interactiontools.get_interaction_sel('hbond', group_a,group_b)
        salt_bridge_selection = interactiontools.get_interaction_sel('salt_bridge', group_a, group_b)

    data = {
        'frame': [], 'n_cont': [], 'cont_a': [], 'cont_b': [], 'q_values': [] 
    }

    if bool_bonds:
        data['cation_pi'] = []
        data['pi_stacking'] = []
        data['hbond'] = []
        data['salt_bridge'] = []

    for ts in u.trajectory:
        frm = ts.frame
        dims = ts.dimensions
        frm_abs = k * chunk_size + frm

        cm_A = group_a.center_of_mass(wrap=True, unwrap=False, compound='group')
        cm_B = group_b.center_of_mass(wrap=True, unwrap=False, compound='group')
        dist_AB = contacts.distance_array(cm_A[None, :], cm_B[None, :], box=dims)[0, 0]
        
        if dist_AB < d_max * dims[2]:
            dist = contacts.distance_array(group_a, group_b, box=dims)
            mat_contacts = contacts.contact_matrix(dist, radius)
            n_contacts = mat_contacts.sum()

            if n_contacts > 0:
                pairs = np.array(np.where(mat_contacts))

                cont_a_ids_u = group_a.atoms.ids[pairs[0, :]]
                cont_b_ids_u = group_b.atoms.ids[pairs[1, :]]
                cont_a_rids_u = group_a.atoms.resids[pairs[0, :]]
                cont_b_rids_u = group_b.atoms.resids[pairs[1, :]]

                data['frame'].append(frm_abs)
                data['n_cont'].append(n_contacts)
                data['cont_a'].append(cont_a_ids_u.tolist())
                data['cont_b'].append(cont_b_ids_u.tolist())
                
                # Native contacts (q-values) calculation
                q_values_dict = {}
                if bool_q: 
                    pattern = os.path.join(output_path, "*_reference.pkl")
                    matching_files = glob.glob(pattern)
    
                    for file in matching_files:
                        ref_val = pd.read_pickle(file)
                        try:                        
                            ind_nat_A = np.asarray(ref_val["nc_a_ind"])
                            ind_nat_B = np.asarray(ref_val["nc_b_ind"])
                            r0 = np.asarray(ref_val["nc_dist"]).flatten()
                            prots = ref_val["prots"]
                            #print("selections", len(group_a), len(group_b), flush=True)
        
                            if len(ind_nat_A) == 0 or len(ind_nat_B) == 0:
                                print(f"Skipping {prots}: empty reference contact set")
                                continue
                    
                            if np.max(ind_nat_A) >= dist.shape[0] or np.max(ind_nat_B) >= dist.shape[1]:
                                #print(f"Skipping {prots}: indices out of bounds for current group")
                                continue
                                
                            r = dist[ind_nat_A, ind_nat_B]
                            q = np.mean(1.0 / (1 + np.exp(BETA_CONST * (r - LAMBDA_CONST * r0))))
                            q_values_dict[prots] = float(q)
                        except:
                            print('no ' + ref_val["prots"] + ' interface')
                data['q_values'].append(q_values_dict)

                # Interaction profiles 
                if bool_bonds:
                    l1 = list(set(cont_a_rids_u))
                    l2 = list(set(cont_b_rids_u))
                    all_cont_resids = l1 + l2

                    all_chains = (group_a + group_b)
                    all_resids = all_chains.residues.resids

                    def eval_candidate_pairs(candiate_pairs, all_cont_resids, all_chains, all_resids):
                        """
                        Filter candidate atom pairs to ensure they belong to residues in contact.
                        
                        Parameters
                        ----------
                        candiate_pairs : array-like
                        all_cont_resids : list
                        all_chains : AtomGroup
                        all_resids : array-like
                        
                        Returns
                        -------
                        numpy.ndarray
                        """
                        mask_subset = np.isin(np.array(candiate_pairs), all_cont_resids).all(axis=1)
                        filtered_pairs = np.array(candiate_pairs)[mask_subset]

                        indices_a = np.searchsorted(all_resids, np.array(filtered_pairs)[:,0])
                        indices_b = np.searchsorted(all_resids, np.array(filtered_pairs)[:,1])
                        mols_a = all_chains.residues[indices_a]
                        mols_b = all_chains.residues[indices_b]
                        mol_pairs = np.column_stack((mols_a, mols_b))

                        return mol_pairs

                    #print(cation_pi_selection)
                    cation_pi_candidates = eval_candidate_pairs(cation_pi_selection, all_cont_resids, all_chains, all_resids)
                    cation_pi_contacts = interactiontools.cation_pi_contact(cation_pi_candidates, distance_cutoff=6.0, angle_cutoff=60.0)

                    #print(pi_stacking_selection)
                    pi_stacking_candidates = eval_candidate_pairs(pi_stacking_selection, all_cont_resids, all_chains, all_resids)
                    pi_stacking_contacts = interactiontools.pi_stacking_contact(pi_stacking_candidates, distance_cutoff=7.0, angle_cutoff=30.0, psi_cutoff=45.0)

                    #print(salt_bridge_selection)
                    salt_bridge_candidates = eval_candidate_pairs(salt_bridge_selection, all_cont_resids, all_chains, all_resids)
                    salt_bridge_contacts = interactiontools.salt_bridge_contact(salt_bridge_candidates, distance_cutoff=4.0)

               
                    #hbond #todo
                    #for donor -h
                    #mask_30 = np.isin(l_sel3[0].ids, cont_a_ids_u)

                    #for hydrogens -h

                    #atoms in contact (cont_a_ids_u) but from atom selection including hydrogens (u_group_a_sel)
                    #mask_30_c = np.isin(u_group_a_sel.ids, cont_a_ids_u)

                    #all atoms of the corresponding residues
                    #group_a_c=u_group_a_sel[mask_30_c].residues.atoms

                    #but only take the hydrogens from group_a_c
                    #group_c=group_a_c.select_atoms(group_a_sc_H)

                    #for aceptors -
                    #mask_31 = np.isin(l_sel3[1].ids, cont_b_ids_u)

                    
                    #hbond for later
                    #bool_hbond =interactiontools.hbond_contact(l_sel3[0][mask_30], group_c, l_sel3[1][mask_31], distance_cutoff=3.5, angle_cutoff=30.0)


                    data['cation_pi'].append(cation_pi_contacts)
                    data['pi_stacking'].append(pi_stacking_contacts)
                    data['hbond'].append([]) 
                    data['salt_bridge'].append(salt_bridge_contacts)
                    
    if bool_debug:
        print(f"[{worker_id}] RSS before trim: {psutil.Process().memory_info().rss / 1024 ** 2:.2f} MB", flush=True)

    #save memory
    #del u, group_a, group_b, ts
    #release_memory()
    
    if bool_debug:
        print(f"{worker_id.memory_manager.memory_limit / 1024**3:.2f} GB", flush=True)
        print(f"[{worker_id}] memory after processing {traj}: {psutil.Process().memory_info().rss / 1024**2:.2f} MB", flush=True)

    
    if not data['frame']:
        return empty_contact_df()

    return pd.DataFrame(data)  

In [5]:
#from GMX_models.py_tools.t_2026_01_calc_contacts_doc_sort import process_frame_chunk_for_split as cc

In [6]:
#original data
top='/lustre/miifs01/project/m2_trr146/kugaurav/MUT16_FFR_atomistic/replica_6/dynamics/postprocessing/MUT16_FFR_atm_analysis_MUT16_65_46af39bf2445d01c22c25aadf9d305c4_pi_clean.gro'
xtc='/lustre/miifs01/project/m2_trr146/kugaurav/MUT16_FFR_atomistic/replica_6/dynamics/postprocessing/MUT16_FFR_atm_analysis_MUT16_65_46af39bf2445d01c22c25aadf9d305c4_full_pi_ref.xtc'
sys_domains=pd.read_parquet('/lustre/miifs01/project/m2_trr146/kugaurav/MUT16_FFR_atomistic/replica_6/dynamics/postprocessing/MUT16_FFR_atm_analysis_MUT16_65_46af39bf2445d01c22c25aadf9d305c4_sys_domains.parquet')

In [7]:
u = mda.Universe(top)
traj=top

### all pairs

In [8]:
ALL_MOLS_SEL = [f"resid {row['min']}-{row['max']}" for _, row in sys_domains.iterrows()]
combinations_in = list(itertools.combinations(enumerate(ALL_MOLS_SEL), 2))
#len(combinations_in)

In [9]:
d_max=5
radius=10
k=0
chunk_size=0

In [10]:
l_df=[]
k=0

In [11]:
for a, b in combinations_in:
    print(k)
    group_a_sel=a[1]
    group_b_sel=b[1]
    l_df.append(process_frame_chunk_for_split(top, traj, group_a_sel, group_b_sel, d_max, radius, k, chunk_size))
    k=k+1

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [15]:
full_df=pd.concat(l_df, ignore_index=True)
full_df

,frame,n_cont,cont_a,cont_b,q_values,cation_pi,pi_stacking,hbond,salt_bridge
0,0,704,"[2358, 2360, 2360, 2360, 2360, 2360, 2360, 236...","[9747, 9745, 9747, 9928, 9929, 9931, 9933, 974...",{},[],[],[],[]
1,0,11201,"[1394, 1395, 1395, 1395, 1397, 1397, 1397, 139...","[12685, 12570, 12573, 12685, 12568, 12570, 125...",{},[],[],[],[]
2,0,536,"[1933, 1933, 1950, 1950, 1950, 1951, 1951, 195...","[16929, 16930, 16928, 16929, 16930, 16928, 169...",{},[],[],[],[]
3,0,3338,"[297, 299, 299, 299, 301, 301, 301, 301, 301, ...","[26743, 26740, 26743, 26746, 26739, 26740, 267...",{},[],[],[],[]
4,0,14131,"[567, 567, 567, 567, 569, 569, 569, 569, 573, ...","[30673, 30675, 30679, 30689, 30673, 30675, 306...",{},"[[103, 2055], [104, 2055]]",[],[],"[[104, 2053], [1995, 97]]"
...,...,...,...,...,...,...,...,...,...
750,0,45,"[54239, 54247, 54247, 54247, 54247, 54247, 542...","[62219, 62218, 62219, 62220, 62235, 62237, 622...",{},[],[],[],[]
751,0,7978,"[51729, 51729, 51729, 51733, 51733, 51733, 517...","[68823, 68825, 68834, 68803, 68819, 68821, 688...",{},[],[],[],[]
752,0,100,"[54672, 54672, 54679, 54826, 54826, 54828, 548...","[59561, 59571, 59571, 59554, 59556, 59552, 595...",{},[],[],[],[]
753,0,13717,"[57076, 57096, 57096, 57096, 57096, 57097, 570...","[62363, 62357, 62358, 62359, 62363, 62357, 623...",{},[],[],[],[]


In [16]:
full_df.columns

Index(['frame', 'n_cont', 'cont_a', 'cont_b', 'q_values', 'cation_pi',
       'pi_stacking', 'hbond', 'salt_bridge'],
      dtype='object')

In [17]:
#load data frame
full_df[['frame', 'n_cont', 'cont_a', 'cont_b', 'cation_pi',
       'pi_stacking', 'hbond', 'salt_bridge']].to_parquet('./combined_data2.parquet', index=False)

In [18]:
#full_df=pd.read_parquet('./combined_data.parquet')
#full_df

In [19]:
pi_stacking = [
    list(outer_item) 
    for outer_item in full_df['pi_stacking'].values 
    if len(outer_item) > 0
]

In [20]:
pi_stacking

[[[730, 7395], [774, 7334]],
 [[1064, 2643]],
 [[1169, 9583]],
 [[1676, 4600]],
 [[1711, 10265]],
 [[1891, 8373]],
 [[1910, 6599]],
 [[2622, 7712]],
 [[2869, 7352]],
 [[2770, 10787]],
 [[3213, 4084]],
 [[3185, 5399]],
 [[3557, 7829]],
 [[5175, 11042]],
 [[5476, 5593]],
 [[5512, 10662], [5536, 10602]],
 [[5820, 9475]],
 [[6130, 6278]],
 [[6234, 9150]],
 [[6664, 8040]],
 [[6625, 8565]],
 [[7387, 9104]],
 [[8221, 8806]],
 [[8986, 11071]]]

In [21]:
l_pi_stacking=[]
for item in pi_stacking:
    for i in item:
        l_pi_stacking.append(list(i))

l_pi_stacking

[[730, 7395],
 [774, 7334],
 [1064, 2643],
 [1169, 9583],
 [1676, 4600],
 [1711, 10265],
 [1891, 8373],
 [1910, 6599],
 [2622, 7712],
 [2869, 7352],
 [2770, 10787],
 [3213, 4084],
 [3185, 5399],
 [3557, 7829],
 [5175, 11042],
 [5476, 5593],
 [5512, 10662],
 [5536, 10602],
 [5820, 9475],
 [6130, 6278],
 [6234, 9150],
 [6664, 8040],
 [6625, 8565],
 [7387, 9104],
 [8221, 8806],
 [8986, 11071]]

In [22]:
cation_pi = [
    list(outer_item) 
    for outer_item in full_df['cation_pi'].values 
    if len(outer_item) > 0
]
cation_pi

[[[103, 2055], [104, 2055]],
 [[5424, 67]],
 [[92, 6568]],
 [[9392, 63]],
 [[132, 10827], [130, 10834], [103, 10824]],
 [[984, 289]],
 [[296, 1590]],
 [[3016, 267]],
 [[296, 4582]],
 [[10068, 316], [275, 10071]],
 [[474, 4879]],
 [[447, 6011], [449, 6011], [5951, 467], [5953, 467], [448, 6011]],
 [[6983, 514], [6984, 514]],
 [[476, 8173], [8214, 507], [8208, 504]],
 [[8552, 676], [619, 8572], [8552, 679]],
 [[780, 2406]],
 [[2857, 696]],
 [[792, 8598]],
 [[992, 6363], [992, 6355]],
 [[964, 9183], [963, 9233]],
 [[952, 10211]],
 [[1328, 2353]],
 [[5120, 1341], [5120, 1314], [5080, 1271]],
 [[5252, 1293]],
 [[8877, 1348]],
 [[1307, 9061], [1308, 9061], [1309, 9011]],
 [[1334, 10362]],
 [[1652, 4600], [4564, 1637], [1640, 4582], [4602, 1676]],
 [[1678, 6309], [1680, 6329], [1680, 6309], [1652, 6309], [1680, 6315]],
 [[1823, 4730], [1825, 4730]],
 [[1824, 8400]],
 [[1852, 8728]],
 [[5435, 2002]],
 [[9380, 2009]],
 [[2194, 6492], [2168, 6487]],
 [[2194, 6673], [6641, 2208]],
 [[8017, 2187]]

In [23]:
l_cation_pi=[]
for item in cation_pi:
    for i in item:
        l_cation_pi.append(list(i))

In [24]:
l_cation_pi

[[103, 2055],
 [104, 2055],
 [5424, 67],
 [92, 6568],
 [9392, 63],
 [132, 10827],
 [130, 10834],
 [103, 10824],
 [984, 289],
 [296, 1590],
 [3016, 267],
 [296, 4582],
 [10068, 316],
 [275, 10071],
 [474, 4879],
 [447, 6011],
 [449, 6011],
 [5951, 467],
 [5953, 467],
 [448, 6011],
 [6983, 514],
 [6984, 514],
 [476, 8173],
 [8214, 507],
 [8208, 504],
 [8552, 676],
 [619, 8572],
 [8552, 679],
 [780, 2406],
 [2857, 696],
 [792, 8598],
 [992, 6363],
 [992, 6355],
 [964, 9183],
 [963, 9233],
 [952, 10211],
 [1328, 2353],
 [5120, 1341],
 [5120, 1314],
 [5080, 1271],
 [5252, 1293],
 [8877, 1348],
 [1307, 9061],
 [1308, 9061],
 [1309, 9011],
 [1334, 10362],
 [1652, 4600],
 [4564, 1637],
 [1640, 4582],
 [4602, 1676],
 [1678, 6309],
 [1680, 6329],
 [1680, 6309],
 [1652, 6309],
 [1680, 6315],
 [1823, 4730],
 [1825, 4730],
 [1824, 8400],
 [1852, 8728],
 [5435, 2002],
 [9380, 2009],
 [2194, 6492],
 [2168, 6487],
 [2194, 6673],
 [6641, 2208],
 [8017, 2187],
 [2168, 8556],
 [2169, 8556],
 [2156, 9523]

In [ ]:
##check elemtns

In [25]:
#pi stacking manual
residue_pairs = [
    [787, 2770], [3213, 4084], [6723, 6726], [9279, 9287],
    [1910, 6599], [2869, 7352], [3130, 3138], [9475, 5820],
    [5536, 602], [7829, 3557], [7739, 7738], [774, 7334],
    [5051, 5055], [18, 8], [1020, 1030], [5175, 1042],
    [3354, 3391], [9492, 9494], [9150, 6234], [730, 7395],
    [6625, 8565], [6664, 8040], [6130, 6278], [265, 1711],
    [7067, 7147], [6355, 6363], [6707, 6706], [5399, 3185],
    [9583, 1169], [1094, 1097], [827, 824], [2015, 2029],
    [2622, 7712], [5476, 5593], [6775, 6788], [531, 534],
    [5512, 662], [9104, 7387], [1064, 2643], [4600, 1676],
    [7395, 7394], [8221, 8806], [4652, 4662], [8986, 1071],
    [2643, 2750], [4463, 4460], [4208, 4223], [1891, 8373]
]

In [27]:
#pi_stacking
l_not_pi_stacking = []

for item in l_pi_stacking:

    if [item[0], item[1]] in residue_pairs or [item[1], item[0]] in residue_pairs:
        #print("found")
        continue

    else:
        print("No matches")
        str_a=f"resid {item[0]}"
        str_b=f"resid {item[1]}"
        print(u.select_atoms(str_a).residues.resids, u.select_atoms(str_b).residues.resids)
        print(u.select_atoms(str_a).residues.resnames, u.select_atoms(str_b).residues.resnames)
        l_not_pi_stacking.append(item)

No matches
[1711] [10265]
['PHE'] ['TYR']
No matches
[2770] [10787]
['TYR'] ['PHE']
No matches
[5175] [11042]
['TYR'] ['TYR']
No matches
[5512] [10662]
['TYR'] ['TYR']
No matches
[5536] [10602]
['PHE'] ['PHE']
No matches
[8986] [11071]
['TYR'] ['TYR']


In [28]:
#cation pi manual
cat_pi=[[7176, 7180], [6316, 6190], [9380, 2009], [3914, 3912], 
        [9220, 9226], [2857, 4986], [1506, 1504], [6811, 6825], 
        [7004, 7051], [4430, 4428], [10794, 10727], [6984, 3729], 
        [8359, 8495], [9592, 9597], [984, 983], [6838, 6836], 
        [1162, 1155], [6324, 8935], [4430, 4428], [1678, 1692], 
        [8359, 8495], [447, 430], [8176, 8147], [3188, 3191], 
        [984, 289], [9552, 9549], [2712, 2717], [6811, 6825], 
        [276, 267], [7157, 7169], [2500, 3163], [9220, 9226], 
        [6813, 6825], [9380, 9383], [130, 137], [4258, 4265], 
        [132, 10827], [1640, 4582], [2856, 2838], [2844, 2841],
        [5093, 7807], [1506, 1504], [9590, 6481], [10253, 10237], 
        [4258, 4265], [7673, 7685], [1136, 1127], [105, 95], 
        [3704, 3679], [2156, 9523], [3543, 3568], [5424, 9448],
        [2712, 2697], [4604, 6336], [9552, 9549], [5252, 1293], 
        [2856, 4437], [9420, 5708], [3914, 3851], [2856, 4437], 
        [6296, 6352], [2712, 2697], [130, 137], [2500, 2503], 
        [9592, 9588], [3914, 3851], [3704, 3679], [296, 1590], 
        [5424, 67], [2876, 2875], [2500, 2503], [5112, 9007], 
        [1136, 1127], [3200, 5412], [2366, 7361], [3016, 267], 
        [9565, 9540], [5779, 5804], [9246, 6272], [5978, 5992],
        [8176, 8147], [6296, 6352], [5424, 67], [476, 481], 
        [9420, 5710], [2366, 7361], [9392, 63], [10794, 10727], 
        [2339, 8814], [5292, 5288], [3228, 3224], [1680, 6329], 
        [1680, 6329], [608, 660], [4948, 4960], [621, 602], 
        [4060, 4170], [1506, 1486], [2882, 4395], [9420, 5710], 
        [9764, 9718], [1678, 1692], [132, 10827], [1680, 6315], 
        [7348, 7903], [1678, 6309], [2168, 6487], [4602, 4632], 
        [6668, 6673], [964, 9183], [2328, 2331], [11111, 8247],
        [620, 626], [3199, 8001], [10423, 10406], [2856, 2838], 
        [296, 4582], [6295, 6352], [6641, 2208], [8552, 8551], 
        [3543, 3563], [9590, 6481], [963, 9233], [10928, 10925], 
        [6641, 2208], [952, 10211], [5120, 5111], [992, 6355], 
        [9592, 9588], [2341, 10352], [10423, 10406], [5290, 8895],
        [8361, 8495], [8216, 8909], [10928, 10925], [9762, 9769], 
        [3360, 3363], [6150, 9054], [6295, 6352], [2712, 2717], 
        [952, 10211], [10794, 8608], [8520, 8517], [9565, 9540],
        [6666, 6664], [1997, 2029], [6813, 6825], [476, 8173], 
        [2512, 2525], [449, 430], [1984, 1981], [2341, 10352],
        [9420, 9411], [5252, 1293], [3200, 8082], [5462, 5460], 
        [10794, 8608], [5951, 467], [4602, 4632], [6668, 6673], 
        [2512, 2525], [9762, 9769], [2876, 2880], [2540, 3213],
        [9380, 2009], [2857, 4986], [2340, 2270], [1984, 1981],
        [9420, 9411], [2194, 6492], [8531, 2406], [1334, 10362],
        [2882, 4395], [8015, 2536], [5424, 5421], [5424, 5421], 
        [2328, 2331], [8017, 2187], [964, 9183], [11113, 8247], 
        [5120, 5111], [2156, 9523], [3371, 9276], [3016, 267], 
        [1652, 6309], [8531, 2407], [4392, 4389], [449, 430], 
        [11132, 11136], [4392, 4389], [6838, 6836]]

In [29]:
#cation_pi
l_not_cation_pi = []

for item in l_cation_pi:

    if [item[0], item[1]] in cat_pi or [item[1], item[0]] in cat_pi:
        #print("found")
        continue

    else:
        print("No matches")
        str_a=f"resid {item[0]}"
        str_b=f"resid {item[1]}"
        print(u.select_atoms(str_a).residues.resids, u.select_atoms(str_b).residues.resids)
        print(u.select_atoms(str_a).residues.resnames, u.select_atoms(str_b).residues.resnames)
        l_not_cation_pi.append(item)

No matches
[103] [2055]
['ARG'] ['PHE']
No matches
[104] [2055]
['ARG'] ['PHE']
No matches
[92] [6568]
['ARG'] ['PHE']
No matches
[130] [10834]
['ARG'] ['TYR']
No matches
[103] [10824]
['ARG'] ['TYR']
No matches
[10068] [316]
['ARG'] ['PHE']
No matches
[275] [10071]
['ARG'] ['TYR']
No matches
[474] [4879]
['ARG'] ['TYR']
No matches
[447] [6011]
['ARG'] ['PHE']
No matches
[449] [6011]
['ARG'] ['PHE']
No matches
[5953] [467]
['ARG'] ['PHE']
No matches
[448] [6011]
['ARG'] ['PHE']
No matches
[6983] [514]
['ARG'] ['TYR']
No matches
[6984] [514]
['ARG'] ['TYR']
No matches
[8214] [507]
['ARG'] ['PHE']
No matches
[8208] [504]
['LYS'] ['TYR']
No matches
[8552] [676]
['LYS'] ['TYR']
No matches
[619] [8572]
['ARG'] ['PHE']
No matches
[8552] [679]
['LYS'] ['PHE']
No matches
[780] [2406]
['ARG'] ['TYR']
No matches
[2857] [696]
['ARG'] ['TYR']
No matches
[792] [8598]
['ARG'] ['TYR']
No matches
[992] [6363]
['ARG'] ['PHE']
No matches
[1328] [2353]
['LYS'] ['TYR']
No matches
[5120] [1341]
['ARG'] ['T

## minimal specifc

In [ ]:

# Select aromatic residues (ResidueGroup)
aromatic_residues = u.select_atoms("resname PHE TYR").residues

# Build candidate pairs once (by residue identity, not per frame)
candidate_res_pairs = list(combinations(aromatic_residues, 2))
[candidate_res_pairs[0], candidate_res_pairs[184070], candidate_res_pairs[-1]]



aromatic_residues = u.select_atoms("resname PHE TYR").residues
cationic_residues = u.select_atoms("resname LYS ARG").residues

candidate_res_pairs = list(product(cationic_residues,aromatic_residues))

[candidate_res_pairs[0], candidate_res_pairs[540795], candidate_res_pairs[-1]]

#hier muesste die volle kette stehen
#group_a_sel="resid 1064-2000 "
#group_b_sel="resid 2643-5000 "

#hier muesste die volle kette stehen
group_a_sel="resid 9380-9800"
group_b_sel="resid 2009-2100"